# 02 · Descriptive Statistics & Correlation Matrices

**Brazilian Stock-Bond Correlation Study**

This notebook produces the foundational empirical evidence:

1. Regime-split summary statistics (Table 1 of whitepaper)
2. Unconditional full-sample correlation matrix
3. Correlation heatmaps by macro regime
4. Crisis-period asset returns heatmap (Table 2 of whitepaper)
5. Distribution analysis — skewness and fat tails


In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats as scipy_stats

from fetch import load_master, CRISES, REGIMES

master = load_master()

# ── Helpers ───────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150, "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 11,
})

RET_COLS = ["ibov", "ntnb", "ltn", "ntnf", "lft_proxy"]
LABELS = {
    "ibov":      "Ibovespa",
    "ntnb":      "NTN-B 5yr",
    "ltn":       "LTN 2yr",
    "ntnf":      "NTN-F 10yr",
    "lft_proxy": "LFT (CDI)",
}
CRISIS_COLORS = {
    "GFC":"#d62728","Dilma":"#ff7f0e","Joesley":"#9467bd",
    "COVID":"#2ca02c","Americanas":"#8c564b","Fiscal24":"#e377c2",
}

def regime_df(df, name):
    s, e = REGIMES[name]
    return df[(df.index >= s) & (df.index <= e)][RET_COLS].dropna(how="all")

def crisis_df(df, name):
    s, e = CRISES[name]
    return df[(df.index >= s) & (df.index <= e)][RET_COLS].dropna(how="all")

## 1. Regime-split summary statistics

The most important context table: how do returns and volatility differ across Brazil's six macro regimes?
Positive mean returns for both stocks and bonds in the same period = "everyone wins" (monetary dominance).
Negative returns in both = "fiscal dominance / crisis."


In [ ]:
def regime_stats(df, period_name, start, end):
    sub = df[(df.index >= start) & (df.index <= end)][RET_COLS].dropna(how="all") * 100
    rows = []
    for col in RET_COLS:
        s = sub[col].dropna()
        if len(s) < 20:
            continue
        ann_ret = s.mean() * 252
        ann_vol = s.std() * np.sqrt(252)
        # CDI-adjusted Sharpe
        cdi_ann = df[(df.index >= start) & (df.index <= end)]["cdi_level"].mean()
        sharpe  = (ann_ret - cdi_ann) / ann_vol if ann_vol > 0 else np.nan
        rows.append({
            "Regime":   period_name,
            "Asset":    LABELS[col],
            "Ann ret%": round(ann_ret, 1),
            "Ann vol%": round(ann_vol, 1),
            "Sharpe":   round(sharpe, 2),
            "Skew":     round(float(scipy_stats.skew(s)), 2),
            "Kurt":     round(float(scipy_stats.kurtosis(s)), 2),
            "N":        len(s),
        })
    return pd.DataFrame(rows)

all_stats = pd.concat([
    regime_stats(master, name, s, e)
    for name, (s, e) in REGIMES.items()
])

# Pivot for readability
pivot = all_stats.pivot_table(
    index="Asset", columns="Regime",
    values="Ann ret%", aggfunc="first"
)[list(REGIMES.keys())]
print("=== Annualised returns by regime (%) ===")
print(pivot.round(1).to_string())

pivot_vol = all_stats.pivot_table(
    index="Asset", columns="Regime",
    values="Ann vol%", aggfunc="first"
)[list(REGIMES.keys())]
print("\n=== Annualised volatility by regime (%) ===")
print(pivot_vol.round(1).to_string())

all_stats.to_csv("../outputs/tbl_regime_stats.csv", index=False)
print("\nSaved: outputs/tbl_regime_stats.csv")

## 2. Unconditional full-sample correlation matrix

The baseline result. Are stocks and bonds negatively correlated (good diversification)
or positively correlated (Brazil's expected finding)?


In [ ]:
df_ret = master[RET_COLS].dropna(how="all") * 100
corr_full = df_ret.corr(method="pearson")
corr_full.index   = [LABELS[c] for c in corr_full.index]
corr_full.columns = [LABELS[c] for c in corr_full.columns]

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr_full, dtype=bool), k=1)
sns.heatmap(
    corr_full, mask=mask,
    annot=True, fmt=".3f", cmap="RdBu_r",
    vmin=-0.5, vmax=0.5, center=0,
    square=True, linewidths=0.5,
    cbar_kws={"shrink": 0.8, "label": "Pearson ρ"},
    ax=ax,
)
ax.set_title(
    f"Unconditional correlation matrix\n"
    f"Full sample: 2005–2026  (N≈{len(df_ret):,} daily obs)",
    fontsize=12, pad=12,
)
plt.tight_layout()
plt.savefig("../outputs/fig_corr_full_sample.png", dpi=150, bbox_inches="tight")
plt.show()

# Highlight the key finding
ibov_ntnb = corr_full.loc["Ibovespa", "NTN-B 5yr"]
ibov_ltn  = corr_full.loc["Ibovespa", "LTN 2yr"]
ibov_lft  = corr_full.loc["Ibovespa", "LFT (CDI)"]
print(f"Key findings:")
print(f"  Ibovespa vs NTN-B  : ρ = {ibov_ntnb:+.3f}")
print(f"  Ibovespa vs LTN    : ρ = {ibov_ltn:+.3f}")
print(f"  Ibovespa vs LFT    : ρ = {ibov_lft:+.3f}")
print("  (Negative = diversification works; Positive = it doesn't)")

## 3. Regime-split correlation heatmaps

**This is the core empirical contribution**: showing that correlations are not stable across regimes.
The Lula Boom / Reform Era should show lower (possibly negative) Ibovespa–bond correlation.
Fiscal dominance episodes should show strongly positive correlation.


In [ ]:
regime_list = list(REGIMES.keys())
n_regimes   = len(regime_list)
ncols, nrows = 3, 2

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 9))
axes = axes.flatten()

regime_corrs = {}
for i, (name, (s, e)) in enumerate(REGIMES.items()):
    sub = master[(master.index >= s) & (master.index <= e)][RET_COLS].dropna(how="all")
    if len(sub) < 30:
        axes[i].set_visible(False)
        continue

    corr = sub.corr()
    corr.index   = [LABELS[c] for c in corr.index]
    corr.columns = [LABELS[c] for c in corr.columns]
    regime_corrs[name] = corr

    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(
        corr, mask=mask,
        annot=True, fmt=".2f", cmap="RdBu_r",
        vmin=-0.6, vmax=0.6, center=0,
        square=True, linewidths=0.4,
        cbar=False, ax=axes[i], annot_kws={"size": 8},
    )
    n_obs = len(sub)
    axes[i].set_title(f"{name}\n(N={n_obs:,})", fontsize=10, fontweight="bold")
    axes[i].tick_params(labelsize=8)

# Shared colorbar
sm = plt.cm.ScalarMappable(cmap="RdBu_r",
                            norm=plt.Normalize(vmin=-0.6, vmax=0.6))
sm.set_array([])
fig.colorbar(sm, ax=axes, shrink=0.5, label="Pearson ρ", pad=0.02)
fig.suptitle(
    "Stock-bond correlation by macro regime\n"
    "Brazil 2004–2026",
    fontsize=14, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.savefig("../outputs/fig_corr_by_regime.png", dpi=150, bbox_inches="tight")
plt.show()

# Print the Ibovespa row across regimes
print("=== Ibovespa correlations across regimes ===")
for name, corr in regime_corrs.items():
    row = corr.loc["Ibovespa"].drop("Ibovespa")
    vals = "  |  ".join(f"{c}: {v:+.3f}" for c, v in row.items())
    print(f"{name:<25}  {vals}")

## 4. Crisis-period asset returns heatmap

Shows cumulative returns during each crisis across all asset classes.
This is Table 2 of the whitepaper: the "triple whammy" evidence.


In [ ]:
crisis_returns = {}
for name, (s, e) in CRISES.items():
    sub = master[(master.index >= s) & (master.index <= e)][RET_COLS]
    # Cumulative log return → simple total return
    cum = np.exp(sub.sum()) - 1
    crisis_returns[name] = cum * 100  # in %

cr_df = pd.DataFrame(crisis_returns).T
cr_df.columns = [LABELS[c] for c in cr_df.columns]

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    cr_df,
    annot=True, fmt=".1f",
    cmap="RdYlGn", center=0, vmin=-55, vmax=55,
    linewidths=0.6, linecolor="white",
    cbar_kws={"label": "Cumulative return (%)", "shrink": 0.7},
    ax=ax,
)
ax.set_title("Cumulative returns during crisis episodes (%)", fontsize=13, pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("../outputs/fig_crisis_returns_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# Count: how many crises had BOTH Ibovespa AND NTN-B negative?
cr_raw = pd.DataFrame(crisis_returns).T
both_neg = ((cr_raw["ibov"] < 0) & (cr_raw["ntnb"] < 0)).sum()
print(f"\nCrises with BOTH Ibovespa AND NTN-B negative: {both_neg}/{len(cr_raw)}")
print("(= number of 'triple whammy' episodes)")
print(cr_raw[["ibov","ntnb"]].round(1).to_string())

## 5. Return distribution analysis

Tests normality and tail behaviour — motivation for using copulas and ES over Gaussian VaR.


In [ ]:
from scipy.stats import jarque_bera, normaltest

fig, axes = plt.subplots(1, len(RET_COLS), figsize=(16, 4))

print("=== Normality tests (p < 0.05 = reject normality) ===")
for i, col in enumerate(RET_COLS):
    s = master[col].dropna() * 100

    # QQ plot vs normal
    (qt, ql) = scipy_stats.probplot(s, dist="norm")[:2]
    axes[i].scatter(qt[0], qt[1], s=2, alpha=0.4, color="#1f77b4")
    axes[i].plot(qt[0], qt[0] * ql[0] + ql[1], color="#d62728", lw=1.5)
    axes[i].set_title(LABELS[col], fontsize=9.5)
    axes[i].set_xlabel("Theoretical quantiles", fontsize=8)
    if i == 0: axes[i].set_ylabel("Sample quantiles", fontsize=8)

    jb_stat, jb_p = jarque_bera(s)
    print(f"  {LABELS[col]:<25} JB stat={jb_stat:8.1f}  p={jb_p:.2e}  "
          f"skew={scipy_stats.skew(s):+.3f}  "
          f"exc.kurt={scipy_stats.kurtosis(s):.2f}")

fig.suptitle("Q-Q plots vs normal distribution
"
             "(deviation in tails = fat tails / non-normality)", fontsize=12)
plt.tight_layout()
plt.savefig("../outputs/fig_qq_plots.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Scatter matrix: pairwise return relationships

Visualises nonlinear dependencies and asymmetric tail behaviour — motivation for copulas.


In [ ]:
df_plot = master[RET_COLS].dropna(how="all") * 100
df_plot.columns = [LABELS[c] for c in df_plot.columns]

# Colour by crisis vs non-crisis
colors_scatter = master["crisis"].map(
    lambda x: CRISIS_COLORS.get(x, "#aaaaaa55")
).reindex(df_plot.index)

pg = sns.PairGrid(df_plot, diag_sharey=False)
pg.map_diag(sns.histplot, bins=50, color="#1f77b4", alpha=0.7)
pg.map_lower(sns.scatterplot, s=2, alpha=0.3,
             hue=colors_scatter.values, palette=None, legend=False)
pg.map_upper(lambda x, y, **kw: None)  # skip upper triangle

for i in range(len(RET_COLS)):
    for j in range(len(RET_COLS)):
        if i < j:
            ax = pg.axes[i, j]
            xc = df_plot.columns[j]
            yc = df_plot.columns[i]
            r  = df_plot[xc].corr(df_plot[yc])
            ax.text(0.5, 0.5, f"ρ={r:+.3f}", transform=ax.transAxes,
                    ha="center", va="center", fontsize=11,
                    fontweight="bold",
                    color="#d62728" if r > 0 else "#2ca02c")
            ax.set_visible(True)
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values(): spine.set_visible(False)

pg.figure.suptitle("Pairwise return scatter matrix
"
                   "(red dots = crisis periods, upper triangle = Pearson ρ)",
                   y=1.01, fontsize=12)
pg.figure.set_size_inches(12, 11)
plt.tight_layout()
plt.savefig("../outputs/fig_scatter_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## ✅ Notebook 02 complete

**Key findings:**

| Finding                                         | Implication                                                                                          |
| ----------------------------------------------- | ---------------------------------------------------------------------------------------------------- |
| Full-sample ρ(Ibovespa, NTN-B) > 0              | Bonds don't hedge equities on average — confirming Brazil as a permanent positive-correlation market |
| Correlations are strongly regime-dependent      | Time-varying methods (DCC-GARCH) are necessary                                                       |
| Crisis-period heatmap shows simultaneous losses | "Triple whammy" — stocks, bonds, AND currency sell off together                                      |
| Jarque-Bera strongly rejects normality          | Gaussian VaR understates tail risk; copulas + ES are appropriate                                     |

**Outputs saved:**

- `fig_corr_full_sample.png` — Figure 2 (whitepaper)
- `fig_corr_by_regime.png` — Figure 3 (whitepaper)
- `fig_crisis_returns_heatmap.png` — Table 2 (whitepaper)
- `tbl_regime_stats.csv` — Table 1 (whitepaper)

**Next:** `03_rolling_corr.ipynb` — time-varying rolling correlations + structural break tests
